In [2]:
# ==============================================================
# MULTI-DATASET INTENT: A1..A15 -> 4 DATASETS (+ OTHER)
# Train per-dataset: IndoBERT & RoBERTa-base, then MLP ensemble
# Save models + metrics per dataset
# ==============================================================

import os, json, random, shutil, math
from collections import Counter, defaultdict

import numpy as np
import torch
from torch import nn
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------- 0) PATH DATA ----------
TRAIN_JSON = "/workspace/train_anomaly_general_expanded (1).jsonl"   # ganti ssuai kamu
VALID_JSON = "/workspace/valid_anomaly_general_expanded.jsonl"   # ganti sesuai kamu

# ---------- 1) DEFINISI GROUP ----------
# 4 dataset sbb:
# D1: A1, A2, A3, A4, OTHER
# D2: A5, A6, A7, OTHER
# D3: A8, A9, A10, OTHER
# D4: A11, A12, A13, A14, A15, OTHER
GROUPS = {
    "D1": ["A1","A2","A3","A4"],
    "D2": ["A5","A6","A7"],
    "D3": ["A8","A9","A10"],
    "D4": ["A11","A12","A13","A14","A15"],
}

# ---------- 2) UTIL: LOAD & FILTER ke 4 DATASET + OTHER ----------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_all = load_jsonl(TRAIN_JSON)
valid_all = load_jsonl(VALID_JSON)

labels_all = sorted({r["label"] for r in train_all})
print("All labels in training:", labels_all)

def build_dataset_for_group(name, targets, train_src, valid_src):
    def map_rows(rows):
        out = []
        for r in rows:
            y = r["label"]
            y2 = y if y in targets else "OTHER"
            out.append({"text": r["text"], "label": y2})
        return out

    tr = map_rows(train_src)
    va = map_rows(valid_src)

    # label set
    labels = sorted(list(set([r["label"] for r in tr])))
    # ensure all target labels + OTHER exist
    want = set(targets) | {"OTHER"}
    for w in want:
        if w not in labels:
            labels.append(w)
    labels = sorted(labels)

    return tr, va, labels

# ---------- 3) TRAIN/EVAL HELPER ----------
def preprocess_fn(tokenizer, label2id):
    def _pp(batch):
        enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
        enc["labels"] = [label2id[y] for y in batch["label"]]
        return enc
    return _pp

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "acc": accuracy_score(y, preds),
        "f1_macro": f1_score(y, preds, average="macro")
    }

def train_one(model_name, out_dir, ds_dict, id2label, label2id, epochs=8, lr=3e-5, batch=16):
    tok = AutoTokenizer.from_pretrained(model_name)
    enc_ds = ds_dict.map(preprocess_fn(tok, label2id), batched=True, remove_columns=["text","label"])
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id
    )

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=50,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc_ds["train"],
        eval_dataset=enc_ds["validation"],
        tokenizer=tok,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    trainer.train()
    eval_res = trainer.evaluate()

    # Per-class report (pakai prediksi eval set)
    preds = np.argmax(trainer.predict(enc_ds["validation"]).predictions, axis=1)
    y_true = enc_ds["validation"]["labels"]
    report = classification_report(y_true, preds, target_names=[id2label[i] for i in range(len(id2label))], digits=4)
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(report)

    # Simpan label map
    with open(os.path.join(out_dir, "label_map.json"), "w") as f:
        json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

    return model, tok, enc_ds, eval_res

def extract_logits(model, enc_set):
    device = next(model.parameters()).device
    dl = torch.utils.data.DataLoader(enc_set.with_format("torch"), batch_size=32)
    model.eval()
    logits_list, labels_list = [], []
    for batch in dl:
        with torch.no_grad():
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }
            if "token_type_ids" in batch:
                inputs["token_type_ids"] = batch["token_type_ids"].to(device)
            out = model(**inputs)
        logits_list.append(out.logits.detach().cpu())
        labels_list.append(batch["labels"])
    return torch.cat(logits_list), torch.cat(labels_list)

class EnsembleMLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
    def forward(self, x): return self.net(x)

def train_ensemble(logits_a, logits_b, y, out_dir, epochs=30, lr=1e-3):
    X = torch.cat([logits_a, logits_b], dim=1).float()
    y = y.long()
    n = int(0.8*len(X))
    X_tr, X_te = X[:n], X[n:]
    y_tr, y_te = y[:n], y[n:]

    model = EnsembleMLP(X.shape[1], int(y.max().item()+1))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train(); opt.zero_grad()
        out = model(X_tr)
        loss = crit(out, y_tr)
        loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(X_te).argmax(dim=1)
    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(y_te.numpy(), pred.numpy())
    f1  = f1_score(y_te.numpy(), pred.numpy(), average="macro")

    # save ensemble weights
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(out_dir, "ensemble_mlp.pt"))

    return {"acc": acc, "f1_macro": f1}, model

# ---------- 4) PIPELINE UTAMA ----------
ROOT_OUT = "./runs_multi"  # semua artefak akan ke sini
os.makedirs(ROOT_OUT, exist_ok=True)

summary = {}

for dname, targets in GROUPS.items():
    print(f"\n========= {dname}: targets={targets} + OTHER =========")
    tr_rows, va_rows, label_list = build_dataset_for_group(dname, targets, train_all, valid_all)

    # DatasetDict HF
    ds_dict = DatasetDict({
        "train": Dataset.from_list(tr_rows),
        "validation": Dataset.from_list(va_rows),
    })

    # Map label -> id (urut alfabetis)
    labels = sorted(list(set([r["label"] for r in tr_rows])))
    label2id = {l:i for i,l in enumerate(labels)}
    id2label = {i:l for l,i in label2id.items()}
    print("Labels:", labels, " class counts (train):", Counter([r["label"] for r in tr_rows]))

    OUT_DIR = os.path.join(ROOT_OUT, dname)
    os.makedirs(OUT_DIR, exist_ok=True)

    # ---- Latih IndoBERT
    indo_dir = os.path.join(OUT_DIR, "indobert")
    model_i, tok_i, enc_ds_i, eval_i = train_one(
        "indobenchmark/indobert-base-p1", indo_dir, ds_dict, id2label, label2id, epochs=8
    )

    # ---- Latih RoBERTa
    rob_dir = os.path.join(OUT_DIR, "roberta_base")
    model_r, tok_r, enc_ds_r, eval_r = train_one(
        "roberta-base", rob_dir, ds_dict, id2label, label2id, epochs=8
    )

    # ---- Ensemble (pakai logits VALID)
    log_i, y_val = extract_logits(model_i, enc_ds_i["validation"])
    log_r, _     = extract_logits(model_r, enc_ds_r["validation"])
    ens_dir = os.path.join(OUT_DIR, "ensemble_mlp")
    ens_metrics, ens_model = train_ensemble(log_i, log_r, y_val, ens_dir)

    # Simpan ringkasan metrik
    summary[dname] = {
        "labels": labels,
        "indobert": eval_i,
        "roberta": eval_r,
        "ensemble": ens_metrics
    }

# ---------- 5) CETAK RINGKASAN ----------
print("\n================= SUMMARY =================")
for d, stats in summary.items():
    print(f"\n{d} labels={stats['labels']}")
    i = stats["indobert"]; r = stats["roberta"]; e = stats["ensemble"]
    print(f"IndoBERT   -> acc={i['eval_acc']:.4f}, f1={i['eval_f1_macro']:.4f}")
    print(f"RoBERTa    -> acc={r['eval_acc']:.4f}, f1={r['eval_f1_macro']:.4f}")
    print(f"EnsembleMLP-> acc={e['acc']:.4f}, f1={e['f1_macro']:.4f}")

# ---------- 6) CATATAN OUTPUT ----------
# Struktur folder:
# runs_multi/
#   D1/
#     indobert/           (HF save + classification_report.txt + label_map.json)
#     roberta_base/       (HF save + classification_report.txt + label_map.json)
#     ensemble_mlp/ensemble_mlp.pt
#   D2/ ...
#   D3/ ...
#   D4/ ...


All labels in training: ['A1', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9']

========= D1: targets=['A1', 'A2', 'A3', 'A4'] + OTHER =========
Labels: ['A1', 'A2', 'A3', 'A4', 'OTHER']  class counts (train): Counter({'OTHER': 715, 'A2': 65, 'A4': 65, 'A1': 65, 'A3': 65})


Map:   0%|          | 0/975 [00:00<?, ? examples/s]

Map:   0%|          | 0/195 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_808/2504936154.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.907600,0.561255,0.835897,0.527641
2,0.531600,0.393876,0.876923,0.697913
3,0.387100,0.341153,0.882051,0.694860
4,0.358000,0.326248,0.892308,0.717479
5,0.326900,0.336201,0.887179,0.703476
6,0.294200,0.344834,0.897436,0.738868
7,0.278200,0.361299,0.892308,0.707668
8,0.262300,0.367431,0.892308,0.736254


...

Map:   0%|          | 0/195 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_808/2504936154.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,1.152300,0.925099,0.666667,0.133333
2,0.934500,0.967147,0.666667,0.133333
3,0.952200,0.911369,0.666667,0.133333
4,0.938100,0.867789,0.666667,0.133333


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



================= SUMMARY =================

D1 labels=['A1', 'A2', 'A3', 'A4', 'OTHER']
IndoBERT   -> acc=0.8974, f1=0.7389
RoBERTa    -> acc=0.8872, f1=0.7094
EnsembleMLP-> acc=0.8974, f1=0.7576

D2 labels=['A5', 'A6', 'A7', 'OTHER']
IndoBERT   -> acc=0.9744, f1=0.9426
RoBERTa    -> acc=0.9795, f1=0.9542
EnsembleMLP-> acc=1.0000, f1=1.0000

D3 labels=['A10', 'A8', 'A9', 'OTHER']
IndoBERT   -> acc=0.9077, f1=0.6653
RoBERTa    -> acc=0.9026, f1=0.6464
EnsembleMLP-> acc=0.9231, f1=0.6782

D4 labels=['A11', 'A12', 'A13', 'A14', 'A15', 'OTHER']
IndoBERT   -> acc=0.6564, f1=0.2705
RoBERTa    -> acc=0.6667, f1=0.1333
EnsembleMLP-> acc=0.6410, f1=0.1302


In [1]:
# ==============================================================
# A-INTENTS FULL PIPELINE
# Normal (multi-class) + Binary-soft per kelas
# Per dataset: IndoBERT, RoBERTa-base, lalu Ensemble MLP
# ==============================================================

import os, json, random
from collections import Counter
import numpy as np
import torch
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

# ---------------- CONFIG ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 1) PATH DATA (GANTI SESUAI PUNYAMU)
TRAIN_JSON = "/workspace/train_anomaly_general_expanded (1).jsonl"   # ganti ssuai kamu
VALID_JSON = "/workspace/valid_anomaly_general_expanded.jsonl"   # ganti sesuai kamu

# 2) SUBSET DATASETS
GROUPS = {
    "D1": ["A1","A2","A3","A4"],
    "D2": ["A5","A6","A7"],
    "D3": ["A8","A9","A10"],
    "D4a": ["A11","A12","A13"],
    "D4b": ["A14","A15"],
}

# 3) MODEL NAMES
INDO = "indobenchmark/indobert-base-p1"
ROBERTA = "roberta-base"

# 4) OUTPUT ROOT
ROOT_OUT = "./runs_A_full"
os.makedirs(ROOT_OUT, exist_ok=True)

# ---------------- DATA ----------------
def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

train_all = load_jsonl(TRAIN_JSON)
valid_all = load_jsonl(VALID_JSON)

def build_dataset_for_group(targets, train_src, valid_src):
    def map_rows(rows):
        out = []
        for r in rows:
            lab = r["label"] if r["label"] in targets else "OTHER"
            out.append({"text": r["text"], "label": lab})
        return out
    tr, va = map_rows(train_src), map_rows(valid_src)
    labels = sorted(set(targets) | {"OTHER"})   # pastikan OTHER ada
    return tr, va, labels

# ---------------- HELPERS ----------------
def preprocess(tokenizer, label2id):
    def _pp(batch):
        enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
        enc["labels"] = [label2id[y] for y in batch["label"]]
        return enc
    return _pp

def compute_metrics_mc(p):
    logits, y = p
    preds = logits.argmax(axis=1)
    return {"acc": accuracy_score(y, preds), "f1_macro": f1_score(y, preds, average="macro")}

def train_multiclass(model_name, out_dir, ds_dict, id2label, label2id, epochs=8, lr=3e-5, batch=16):
    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(batch):
        enc = tok(batch["text"], truncation=True, padding="max_length", max_length=128)
        # pastikan integer (untuk CrossEntropy)
        enc["labels"] = [int(label2id[y]) for y in batch["label"]]
        return enc

    # map + paksa format Torch (labels = int64)
    enc_ds = ds_dict.map(_pp, batched=True, remove_columns=["text", "label"])
    # set kolom yang ada
    cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in enc_ds["train"].features:
        cols.append("token_type_ids")
    enc_ds.set_format(type="torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id
    )

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",      # <-- perbaikan: eval_strategy
        save_strategy="epoch",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc_ds["train"],
        eval_dataset=enc_ds["validation"],
        tokenizer=tok,                    # deprecation warning aman
        compute_metrics=compute_metrics_mc,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    trainer.train()
    eval_res = trainer.evaluate()

    # # report per-kelas
    # preds = np.argmax(trainer.predict(enc_ds["validation"]).predictions, axis=1)
    # y_true = enc_ds["validation"]["labels"].numpy()  # sudah torch.Long → numpy int
    # report = classification_report(y_true, preds,
    #                                target_names=[id2label[i] for i in range(len(id2label))],
    #                                digits=4)
    pred_out = trainer.predict(enc_ds["validation"])
    preds   = np.argmax(pred_out.predictions, axis=1)
    y_true  = pred_out.label_ids                     # <- ambil label dari sini (bukan dari dataset column)
    
    report = classification_report(
        y_true, preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        digits=4
    )
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(report)
    # with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
    #     f.write(report)
    with open(os.path.join(out_dir, "label_map.json"), "w") as f:
        json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

    

    

    return model, tok, enc_ds, eval_res


def extract_logits(model, enc_set):
    device = next(model.parameters()).device
    dl = torch.utils.data.DataLoader(enc_set.with_format("torch"), batch_size=32)
    model.eval(); logits_list=[]; y_list=[]
    for b in dl:
        with torch.no_grad():
            inputs = {"input_ids": b["input_ids"].to(device),
                      "attention_mask": b["attention_mask"].to(device)}
            if "token_type_ids" in b: inputs["token_type_ids"] = b["token_type_ids"].to(device)
            out = model(**inputs)
        logits_list.append(out.logits.detach().cpu()); y_list.append(b["labels"])
    return torch.cat(logits_list), torch.cat(y_list)

class EnsembleMLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128,out_dim))
    def forward(self,x): return self.net(x)

def train_ensemble_mc(logits_a, logits_b, y, out_dir, epochs=30, lr=1e-3):
    X = torch.cat([logits_a, logits_b], dim=1).float()
    y = y.long()
    n = int(0.8*len(X)); Xtr,Xte = X[:n],X[n:]; ytr,yte = y[:n],y[n:]
    mlp = EnsembleMLP(X.shape[1], int(y.max().item()+1))
    opt = torch.optim.Adam(mlp.parameters(), lr=lr); crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        mlp.train(); opt.zero_grad(); out=mlp(Xtr); loss=crit(out,ytr); loss.backward(); opt.step()
    mlp.eval(); 
    with torch.no_grad(): pred = mlp(Xte).argmax(dim=1)
    acc = accuracy_score(yte.numpy(), pred.numpy()); f1 = f1_score(yte.numpy(), pred.numpy(), average="macro")
    os.makedirs(out_dir, exist_ok=True); torch.save(mlp.state_dict(), os.path.join(out_dir,"ensemble_mlp.pt"))
    return {"acc":acc, "f1_macro":f1}

# ----- Binary one-vs-rest -----
def make_bin_rows(rows, pos_label):
    return [{"text": r["text"], "label": int(r["label"]==pos_label)} for r in rows]

def train_binary(model_name, out_dir, ds_dict, epochs=6, lr=3e-5, batch=16):
    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(b):
        enc = tok(b["text"], truncation=True, padding="max_length", max_length=128)
        # BCEWithLogits butuh target shape [B, 1] (bukan [B])
        #    jadi bungkus tiap label jadi list satu elemen (float)
        enc["labels"] = [[float(x)] for x in b["label"]]   # <<<<< FIX SHAPE
        return enc

    enc = ds_dict.map(_pp, batched=True, remove_columns=["text", "label"])

    cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in enc["train"].features:
        cols.append("token_type_ids")
    enc.set_format(type="torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="multi_label_classification",  # BCEWithLogits
    )

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",     # <<<<< BENAR (bukan eval_strategy)
        save_strategy="epoch",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        logging_steps=50,
        report_to="none",
    )

    def metrics_bin(p):
        # p.predictions shape: [B, 1] ; p.label_ids shape: [B, 1]
        logits = np.array(p.predictions).reshape(-1)   # [B]
        y      = np.array(p.label_ids).reshape(-1)     # [B]
        prob   = 1 / (1 + np.exp(-logits))
        pred   = (prob >= 0.5).astype(int)
        y_int  = y.astype(int)
        try:
            auc = roc_auc_score(y_int, prob)
        except Exception:
            auc = float("nan")
        return {
            "acc": accuracy_score(y_int, pred),
            "f1_macro": f1_score(y_int, pred, average="macro"),
            "auc": auc,
        }

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc["train"],
        eval_dataset=enc["validation"],
        tokenizer=tok,
        compute_metrics=metrics_bin,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    eval_res = trainer.evaluate()
    return model, tok, enc, eval_res




def extract_bin_logit(model, enc_set):
    device = next(model.parameters()).device
    dl = torch.utils.data.DataLoader(enc_set.with_format("torch"), batch_size=32)
    model.eval(); logs=[]; ys=[]
    for b in dl:
        with torch.no_grad():
            inputs = {"input_ids": b["input_ids"].to(device),
                      "attention_mask": b["attention_mask"].to(device)}
            if "token_type_ids" in b: inputs["token_type_ids"] = b["token_type_ids"].to(device)
            out = model(**inputs)
        logs.append(out.logits.detach().cpu()); ys.append(b["labels"])
    return torch.cat(logs).squeeze(1), torch.cat(ys)  # shape: [N]

class BinaryEnsemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,1))
    def forward(self,x): return self.net(x)  # raw logit

def train_ensemble_bin(logit_a, logit_b, y, out_dir, epochs=30, lr=1e-3):
    X = torch.stack([logit_a, logit_b], dim=1).float()     # [N,2]
    y = y.float().view(-1,1)                               # [N,1]
    n = int(0.8*len(X)); Xtr,Xte = X[:n],X[n:]; ytr,yte = y[:n],y[n:]
    mlp = BinaryEnsemble(); opt=torch.optim.Adam(mlp.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        mlp.train(); opt.zero_grad(); out=mlp(Xtr); loss=crit(out,ytr); loss.backward(); opt.step()
    mlp.eval()
    with torch.no_grad():
        logit = mlp(Xte).squeeze(1); prob = torch.sigmoid(logit).numpy(); pred = (prob>=0.5).astype(int)
    y_true = yte.numpy().astype(int)
    acc = accuracy_score(y_true, pred); f1 = f1_score(y_true, pred, average="macro"); auc = roc_auc_score(y_true, prob)
    os.makedirs(out_dir, exist_ok=True); torch.save(mlp.state_dict(), os.path.join(out_dir,"ensemble_bin.pt"))
    return {"acc":acc, "f1_macro":f1, "auc":auc}

# ---------------- MAIN ----------------
summary = {}

for dname, targets in GROUPS.items():
    print(f"\n=== {dname}: {targets} + OTHER ===")
    tr_rows, va_rows, labels = build_dataset_for_group(targets, train_all, valid_all)
    print("Train class counts:", Counter([r["label"] for r in tr_rows]))
    ds = DatasetDict({
        "train": Dataset.from_list(tr_rows),
        "validation": Dataset.from_list(va_rows),
    })
    label2id = {l:i for i,l in enumerate(labels)}
    id2label = {i:l for l,i in label2id.items()}

    OUT = os.path.join(ROOT_OUT, dname); os.makedirs(OUT, exist_ok=True)

    # ---- (A) NORMAL: MULTI-CLASS ----
    indo_dir = os.path.join(OUT, "indobert_mc")
    rob_dir  = os.path.join(OUT, "roberta_mc")
    model_i, tok_i, enc_i, eval_i = train_multiclass(INDO, indo_dir, ds, id2label, label2id, epochs=8)
    model_r, tok_r, enc_r, eval_r = train_multiclass(ROBERTA, rob_dir, ds, id2label, label2id, epochs=8)

    # Ensemble MLP (multi-class)
    log_i, y_val = extract_logits(model_i, enc_i["validation"])
    log_r, _     = extract_logits(model_r, enc_r["validation"])
    ens_mc_dir = os.path.join(OUT, "ensemble_mc")
    ens_mc = train_ensemble_mc(log_i, log_r, y_val, ens_mc_dir)

    summary[dname] = {"labels": labels, "indobert": eval_i, "roberta": eval_r, "ensemble_mc": ens_mc}

    # ---- (B) BINARY SOFT PER-KELAS (one-vs-rest) + ENSEMBLE ----
    print(f"--- Binary soft detectors for {dname} (one-vs-rest) ---")
    for pos_label in targets:   # hanya target label (bukan OTHER)
        print(f"  [+] {pos_label}")
        tr_bin = make_bin_rows(tr_rows, pos_label)
        va_bin = make_bin_rows(va_rows, pos_label)
        ds_bin = DatasetDict({
            "train": Dataset.from_list(tr_bin),
            "validation": Dataset.from_list(va_bin),
        })

        # IndoBERT binary
        bi_dir = os.path.join(OUT, f"indobert_bin_{pos_label}")
        model_bi, tok_bi, enc_bi, eval_bi = train_binary(INDO, bi_dir, ds_bin)

        # RoBERTa binary
        br_dir = os.path.join(OUT, f"roberta_bin_{pos_label}")
        model_br, tok_br, enc_br, eval_br = train_binary(ROBERTA, br_dir, ds_bin)

        # Ensemble binary (gabung dua logit)
        log_bi, yb = extract_bin_logit(model_bi, enc_bi["validation"])
        log_br, _  = extract_bin_logit(model_br, enc_br["validation"])
        ens_bin_dir = os.path.join(OUT, f"ensemble_bin_{pos_label}")
        ens_bin = train_ensemble_bin(log_bi, log_br, yb, ens_bin_dir)

        # simpan metrik ringkas per label
        with open(os.path.join(OUT, f"metrics_binary_{pos_label}.json"), "w") as f:
            json.dump({"indobert": eval_bi, "roberta": eval_br, "ensemble_bin": ens_bin}, f, indent=2)

# ---------------- SUMMARY PRINT ----------------
print("\n================= SUMMARY =================")
for d, stats in summary.items():
    print(f"\n{d} labels={stats['labels']}")
    i = stats["indobert"]; r = stats["roberta"]; e = stats["ensemble_mc"]
    print(f"IndoBERT   -> acc={i['eval_acc']:.4f}, f1={i['eval_f1_macro']:.4f}")
    print(f"RoBERTa    -> acc={r['eval_acc']:.4f}, f1={r['eval_f1_macro']:.4f}")
    print(f"EnsembleMC -> acc={e['acc']:.4f}, f1={e['f1_macro']:.4f}")

# --------- OUTPUT STRUCTURE ----------
# runs_A_full/
#   D1/
#     indobert_mc/ , roberta_mc/ , ensemble_mc/ensemble_mlp.pt
#     indobert_bin_A1/ , roberta_bin_A1/ , ensemble_bin_A1/ensemble_bin.pt  + metrics_binary_A1.json
#     indobert_bin_A2/ ...
#   D2/ ...
#   D3/ ...
#   D4a/ ...
#   D4b/ ...



=== D1: ['A1', 'A2', 'A3', 'A4'] + OTHER ===
Train class counts: Counter({'OTHER': 715, 'A2': 65, 'A4': 65, 'A1': 65, 'A3': 65})


Map:   0%|          | 0/975 [00:00<?, ? examples/s]

Map:   0%|          | 0/195 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_593/2420446086.py:110: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.907600,0.561255,0.835897,0.527641
2,0.531600,0.393876,0.876923,0.697913
3,0.387100,0.341153,0.882051,0.694860
4,0.358000,0.326248,0.892308,0.717479
5,0.326900,0.336201,0.887179,0.703476
6,0.294200,0.344834,0.897436,0.738868
7,0.278200,0.361299,0.892308,0.707668
8,0.262300,0.367431,0.892308,0.736254


...

Map:   0%|          | 0/975 [00:00<?, ? examples/s]

Map:   0%|          | 0/195 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_593/2420446086.py:242: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro,Auc
1,0.308200,0.291688,0.933333,0.482759,0.658918
2,0.245200,0.256276,0.933333,0.482759,0.767540


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

